In [1]:
pip install transformers torch

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [15]:
def generate(prompt, max_new_tokens=40, num_return_sequences=1, do_sample=False, temperature=1.0):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_return_sequences=num_return_sequences,
        do_sample=do_sample,
        temperature=temperature,
    )
    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

In [16]:
zero_shot_prompt = "Classify the sentiment as positive or negative: The service was slow and the food was cold."

print("===== Zero-Shot Prompting =====")
print("Prompt:", zero_shot_prompt)
print("Output:", generate(zero_shot_prompt)[0])

===== Zero-Shot Prompting =====
Prompt: Classify the sentiment as positive or negative: The service was slow and the food was cold.
Output: negative


In [17]:
few_shot_prompt = """Classify the sentiment as positive or negative.

Review: The staff were friendly and the room was clean.
Sentiment: positive

Review: The product broke after one day of use.
Sentiment: negative

Review: The service was slow and the food was cold.
Sentiment:"""

print("\n===== Few-Shot Prompting =====")
print("Output:", generate(few_shot_prompt)[0])


===== Few-Shot Prompting =====
Output: negative


In [18]:
cot_prompt = (
    "Question: A store had 23 apples. They sold 8 apples and then bought 15 more. "
    "How many apples does the store have now? Let's think step by step."
)

print("\n===== Chain-of-Thought Prompting =====")
print("Output:", generate(cot_prompt, max_new_tokens=60)[0])


===== Chain-of-Thought Prompting =====
Output: The store has 23 - 8 = 14 apples. The store now has 23 + 14 = 58 apples. The answer: 58.


In [19]:
# =========================================================
# 4. TREE-OF-THOUGHT PROMPTING
# =========================================================

question = """
A store had 23 apples.
They sold 8 apples.
Then they bought 15 more apples.
How many apples does the store have now?
"""

tot_prompt = f"""
Imagine three different experts are answering this question.
All experts will write down 1 step of their thinking, then share
it with the group. Then all experts will go on to the next step,
etc. If any expert realises they're wrong at any point then they leave.

The question is:

{question}

Give the final answer after the experts complete their reasoning.
"""

print("\n===== Tree-of-Thought Prompting =====")
print(generate(tot_prompt, max_new_tokens=150))


===== Tree-of-Thought Prompting =====
['The store has 23 + 8 = 42 apples. The store has 42 + 15 = 58 apples. The final answer: 58.']


In [20]:
role_prompt = (
    "You are an experienced restaurant critic. Review the following feedback and give "
    "a one-sentence professional assessment: The service was slow and the food was cold."
)

print("\n===== Role-Based Prompting =====")
print("Output:", generate(role_prompt, max_new_tokens=50)[0])



===== Role-Based Prompting =====
Output: The food was not good. The service was slow and the food was cold.


In [21]:
print("\n===== Summary of All Techniques =====")
print("Zero-shot   :", generate(zero_shot_prompt)[0])
print("Few-shot    :", generate(few_shot_prompt)[0])
print("CoT         :", generate(cot_prompt, max_new_tokens=60)[0])
print("Role-based  :", generate(role_prompt, max_new_tokens=50)[0])



===== Summary of All Techniques =====
Zero-shot   : negative
Few-shot    : negative
CoT         : The store has 23 - 8 = 14 apples. The store now has 23 + 14 = 58 apples. The answer: 58.
Role-based  : The food was not good. The service was slow and the food was cold.
